In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/toledo-ohio-property-tax-delinquency/delinquent_toledo.csv
/kaggle/input/cleaned-delinquent-tax-listing-for-2024/geocode_toledo_addresses.ipynb
/kaggle/input/cleaned-delinquent-tax-listing-for-2024/geocode_toledo_addresses.py
/kaggle/input/cleaned-delinquent-tax-listing-for-2024/cleaned_toledo_addresses.csv
/kaggle/input/delinquent-toledo/deliNQ.code-workspace
/kaggle/input/delinquent-toledo/delinquent_property_housing


In [6]:
# Create a geocoding script as a Jupyter notebook
notebook_content = """\
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Geocode Toledo Property Addresses\\n",
    "This notebook geocodes a CSV of delinquent property addresses using Nominatim (OpenStreetMap)."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import pandas as pd\\n",
    "from geopy.geocoders import Nominatim\\n",
    "from geopy.extra.rate_limiter import RateLimiter\\n",
    "import time"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load the cleaned address data\\n",
    "df = pd.read_csv(\"cleaned_toledo_addresses.csv\")\\n",
    "df[\"full_address\"] = df[\"street_number\"] + \" \" + df[\"street_name\"] + \", \" + df[\"city\"] + \", \" + df[\"state\"]"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Initialize geocoder\\n",
    "geolocator = Nominatim(user_agent=\"toledo_delinquent_properties_geocoder\")\\n",
    "geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Safe geocoding function\\n",
    "def safe_geocode(address):\\n",
    "    try:\\n",
    "        loc = geocode(address)\\n",
    "        return (loc.latitude, loc.longitude) if loc else (None, None)\\n",
    "    except:\\n",
    "        return (None, None)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Apply geocoding\\n",
    "df[[\"latitude\", \"longitude\"]] = df[\"full_address\"].apply(lambda x: pd.Series(safe_geocode(x)))"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Save the geocoded data\\n",
    "df.to_csv(\"geocoded_toledo_addresses.csv\", index=False)\\n",
    "print(\"Saved to geocoded_toledo_addresses.csv\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.11"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}
"""

# Save the notebook to a .ipynb file
notebook_path = "geocode_toledo_addresses.ipynb"
with open(notebook_path, "w") as f:
    f.write(notebook_content)

notebook_path


'geocode_toledo_addresses.ipynb'

In [7]:
import pandas as pd

# Load the CSV file into a DataFrame
path ='/kaggle/input/toledo-ohio-property-tax-delinquency/delinquent_toledo.csv'
del_hous = pd.read_csv(path)

# Rename the columns


del_hous_toledo = del_hous

del_hous_toledo.head()
del_hous_toledo.columns

del_hous_cols = {'Parcel Number':'parcel', 'Owner Name as Listed in County Records':'owner', 'Legal Description  as Listed in County Records':'description','Address as Listed in County Records' :'address', 'Total Under taking at the close of the 2023 Tax Year Collection on July 31, 2024. ':'ttax24'}

del_hous_toledo_ren = del_hous_toledo.rename(columns = del_hous_cols)

display(del_hous_toledo_ren.head())

,parcel,owner,description,address,ttax24
0,01-00034,PRICE WILLIE D,ACKLINS ADDN LOTS 9 &10,1724 UPTON,$465.97
1,01-00157,JAHAN INVESTMENTS LLC,ACKLINS 2ND ADDN LOT 13 N 15...W 29 FT E 57 FT,1845 N SUMMIT ST,$428.81
2,01-00204,FAJARDO TEODOMIRA,ACKLINS 2ND ADDN LOT 23,629 E BROADWAY ST,$598.75
3,01-00321,GIRARDOT JEFFREY,ACKLINS 2ND ADDN LOTS 48 T60 FT,704 PARKER AVE,"$1,240.19"
4,01-00381,GOTAY ELIZABETH,ACKLINS 2ND ADDN LOTS 61 T90 FT,523 POTTER ST,"$1,018.38"


In [8]:

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
import numpy as np

# Sort the DataFrame by owner and address
del_hous_toledo_ren_sorted = del_hous_toledo_ren.sort_values(by=['owner', 'address'])

location = del_hous_toledo_ren_sorted[['parcel','owner', 'address']]
address = location['address'] 
owner = location['owner']
parcel = location['parcel']

location_df = pd.DataFrame(location)
display(location_df.head())

address = location_df['address'] 
owner = location_df['owner']
parcel = location_df['parcel']
location_df.loc[:, 'city'] = 'Toledo'
location_df.loc[:, 'county'] = 'Lucas'
location_df.loc[:, 'state'] = 'Ohio'

location_df
!pip install ace-tools-open
import ace_tools_open as tools

tools.display_dataframe_to_user(name="Location Data", dataframe=location_df.head())

,parcel,owner,address
4621,18-00065,111 SOUTH SUMMIT LLC,60 ELMORA AVE
4696,18-71537,111 SOUTH SUMMIT LLC,60 ELMORA AVE
7154,98-13717,11239 WATERSVILLE STREET L,445 CENTRAL AVE STE 304
5948,36-01651,1294 CONANT LLC,1294 CONANT ST # 400B
5949,36-01653,1294 CONANT LLC,1294 CONANT ST # 400B


Location Data


Loading ITables v2.4.4 from the internet... (need help?)


In [9]:
!pip install geocoder
!pip install geopy
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import time
import os

# Load the cleaned address data
df = pd.read_csv("/kaggle/input/cleaned-delinquent-tax-listing-for-2024/cleaned_toledo_addresses.csv")

import re

def split_address(addr):
    """
    Splits address into (street_number, street_name).
    E.g. '1234 Main St' → ('1234', 'Main St')
    If no number found, returns (None, addr)
    """
    if pd.isnull(addr):
        return (None, None)
    match = re.match(r'^\s*(\d+)\s+(.*)', str(addr).strip())
    if match:
        return match.groups()
    else:
        return (None, addr.strip())

# Example usage (after loading your DataFrame):
df[['street_number', 'street_name']] = df['address'].apply(
    lambda x: pd.Series(split_address(x))
)


import numpy as np

def clean_street_number(val):
    # Remove blanks and nan-like values
    if pd.isnull(val) or str(val).strip() == "" or str(val).strip().lower() == "nan":
        return np.nan
    try:
        # Convert float strings like '1311.0' or float type 1311.0 -> 1311 (int), then string
        return str(int(float(val)))
    except Exception:
        return np.nan

df["street_number"] = df["street_number"].apply(clean_street_number)
df = df.dropna(subset=["street_number", "street_name"])


   
# Fix: Convert floats to int strings (e.g., 1311.0 -> "1311"), handle NaNs
df["street_number"] = df["street_number"].fillna("").astype("Int64").astype(str)
df["street_name"] = df["street_name"].fillna("").astype(str)
df["city"] = df["city"].fillna("").astype(str)
df["state"] = df["state"].fillna("").astype(str)

# Drop rows with missing or bad address data
df = df[~df["street_number"].str.lower().str.contains("nan")]
df = df[~df["street_name"].str.lower().str.contains("nan")]
df = df.dropna(subset=["street_number", "street_name"])

# Combine into full address
df["full_address"] = df["street_number"] + " " + df["street_name"] + ", " + df["city"] + ", " + df["state"]

# Geocoder setup (no timeout here)
geolocator = Nominatim(user_agent="toledo_delinquent_properties_geocoder")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1, swallow_exceptions=True)

# Resume support
if os.path.exists("geocoded_partial.csv"):
    geocoded_df = pd.read_csv("geocoded_partial.csv")
    completed_addresses = set(geocoded_df["full_address"])
    print(f"Resuming: {len(completed_addresses)} already processed.")
else:
    geocoded_df = pd.DataFrame()
    completed_addresses = set()

# Correct safe_geocode with timeout passed to geocode directly
def safe_geocode(address):
    if not isinstance(address, str) or "nan" in address.lower():
        print(f"Skipping invalid address: {address}")
        return None, None
    try:
        loc = geolocator.geocode(address, timeout=15)
        if loc:
            return loc.latitude, loc.longitude
        else:
            print(f"Not found: {address}")
            return None, None
    except Exception as e:
        print(f"Error geocoding '{address}': {e}")
        time.sleep(20)
        return None, None

# Geocode in batches
batch_size = 50
new_rows = []

for idx, row in df.iterrows():
    addr = row["full_address"]
    if addr in completed_addresses:
        continue

    lat, lon = safe_geocode(addr)
    row_data = row.to_dict()
    row_data["latitude"] = lat
    row_data["longitude"] = lon
    new_rows.append(row_data)

    if len(new_rows) >= batch_size:
        new_df = pd.DataFrame(new_rows)
        geocoded_df = pd.concat([geocoded_df, new_df], ignore_index=True)
        geocoded_df.to_csv("geocoded_partial.csv", index=False)
        print(f"Saved {len(geocoded_df)} rows to geocoded_partial.csv")
        new_rows = []

# Final write
if new_rows:
    new_df = pd.DataFrame(new_rows)
    geocoded_df = pd.concat([geocoded_df, new_df], ignore_index=True)
    geocoded_df.to_csv("geocoded_partial.csv", index=False)
    print(f"Final save: {len(geocoded_df)} rows total.")


Resuming: 6351 already processed.


In [10]:
import os
print(os.listdir())

import shutil
shutil.copy("/kaggle/working/geocoded_partial.csv", "/kaggle/working/geo.csv")






['.virtual_documents', 'geomod.csv', 'state.db', 'geocoded_partial.csv', 'geocode_toledo_addresses.ipynb', 'geo.csv']


'/kaggle/working/geo.csv'

In [ ]:
import pandas as pd
import numpy as np
import math

from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import time
import os


geo_df = pd.read_csv('/kaggle/working/geo.csv')
display('GEO_DF',geo_df.head(20))

geomod_df = geo_df.drop(['street_number','street_name','full_address'], axis=1)
display(geomod_df.head())

# Geocoder setup (no timeout here)
geolocator = Nominatim(user_agent="toledo_delinquent_properties_geocoder")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1, swallow_exceptions=True)

# Resume support
if os.path.exists("geomod_df.csv"):
    geocoded_df = pd.read_csv("geomod_df.csv")
    completed_addresses = set(geocoded_df['address','city', 'state'])
    print(f"Resuming: {len(completed_addresses)} already processed.")
else:
    geocoded_df = pd.DataFrame()
    completed_addresses = set()

# Correct safe_geocode with timeout passed to geocode directly
def safe_geocode(address):
    if not isinstance(address, str) or "nan" in address.lower():
        print(f"Skipping invalid address: {address}")
        return None, None
    try:
        loc = geolocator.geocode(address, timeout=15)
        if loc:
            return loc.latitude, loc.longitude
        else:
            print(f"Not found: {address}")
            return None, None
    except Exception as e:
        print(f"Error geocoding '{address}': {e}")
        time.sleep(20)
        return None, None

# Geocode in batches
batch_size = 50
new_rows = []

for idx, row in df.iterrows():
    addr = row["address"]
    if addr in completed_addresses:
        continue

    lat, lon = safe_geocode(addr)
    row_data = row.to_dict()
    row_data["latitude"] = lat
    row_data["longitude"] = lon
    new_rows.append(row_data)

    if len(new_rows) >= batch_size:
        new_df = pd.DataFrame(new_rows)
        geocoded_df = pd.concat([geocoded_df, new_df], ignore_index=True)
        geocoded_df.to_csv("geomod.csv", index=False)
        print(f"Saved {len(geocoded_df)} rows to geomod.csv")
        new_rows = []

# Final write
if new_rows:
    new_df = pd.DataFrame(new_rows)
    geocoded_df = pd.concat([geocoded_df, new_df], ignore_index=True)
    geocoded_df.to_csv("geomod.csv", index=False)
    print(f"Final save: {len(geocoded_df)} rows total.")

import os
print(os.listdir())

import shutil
shutil.copy("/kaggle/working/geomod.csv", "/kaggle/working/geo2.csv")


geo2_df = pd.read_csv("/kaggle/working/geo2.csv")
display(geo2_df)

'GEO_DF'

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,parcel,owner,address,city,county,state,street_number,street_name,full_address,latitude,longitude
0,18-00065,111 SOUTH SUMMIT LLC,60 ELMORA AVE,Toledo,Lucas,Ohio,60.0,ELMORA AVE,"60.0 ELMORA AVE, Toledo, Ohio",NaN,NaN
1,18-71537,111 SOUTH SUMMIT LLC,60 ELMORA AVE,Toledo,Lucas,Ohio,60.0,ELMORA AVE,"60.0 ELMORA AVE, Toledo, Ohio",NaN,NaN
2,98-13717,11239 WATERSVILLE STREET L,445 CENTRAL AVE STE 304,Toledo,Lucas,Ohio,445.0,CENTRAL AVE STE 304,"445.0 CENTRAL AVE STE 304, Toledo, Ohio",NaN,NaN
3,36-01651,1294 CONANT LLC,1294 CONANT ST # 400B,Toledo,Lucas,Ohio,1294.0,CONANT ST # 400B,"1294.0 CONANT ST # 400B, Toledo, Ohio",NaN,NaN
4,36-01653,1294 CONANT LLC,1294 CONANT ST # 400B,Toledo,Lucas,Ohio,1294.0,CONANT ST # 400B,"1294.0 CONANT ST # 400B, Toledo, Ohio",NaN,NaN
5,36-01687,1294 CONANT LLC,1294 CONANT ST # 400B,Toledo,Lucas,Ohio,1294.0,CONANT ST # 400B,"1294.0 CONANT ST # 400B, Toledo, Ohio",NaN,NaN
6,15-00193,1301 ADAMS STREET LLC AN OCOMPANY,1301 N SUMMIT ST,Toledo,Lucas,Ohio,1301.0,N SUMMIT ST,"1301.0 N SUMMIT ST, Toledo, Ohio",41.686941,-83.486491
7,15-50334,1301 ADAMS STREET LLC AN OCOMPANY,1301 N SUMMIT ST,Toledo,Lucas,Ohio,1301.0,N SUMMIT ST,"1301.0 N SUMMIT ST, Toledo, Ohio",41.686941,-83.486491
8,15-50344,1301 ADAMS STREET LLC AN OCOMPANY,1301 N SUMMIT ST,Toledo,Lucas,Ohio,1301.0,N SUMMIT ST,"1301.0 N SUMMIT ST, Toledo, Ohio",41.686941,-83.486491
9,15-50354,1301 ADAMS STREET LLC AN OCOMPANY,1301 N SUMMIT ST,Toledo,Lucas,Ohio,1301.0,N SUMMIT ST,"1301.0 N SUMMIT ST, Toledo, Ohio",41.686941,-83.486491


/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,parcel,owner,address,city,county,state,latitude,longitude
0,18-00065,111 SOUTH SUMMIT LLC,60 ELMORA AVE,Toledo,Lucas,Ohio,NaN,NaN
1,18-71537,111 SOUTH SUMMIT LLC,60 ELMORA AVE,Toledo,Lucas,Ohio,NaN,NaN
2,98-13717,11239 WATERSVILLE STREET L,445 CENTRAL AVE STE 304,Toledo,Lucas,Ohio,NaN,NaN
3,36-01651,1294 CONANT LLC,1294 CONANT ST # 400B,Toledo,Lucas,Ohio,NaN,NaN
4,36-01653,1294 CONANT LLC,1294 CONANT ST # 400B,Toledo,Lucas,Ohio,NaN,NaN


Not found: 445 CENTRAL AVE STE 304
Error geocoding '2144 57TH ST': Non-successful status code 503
Error geocoding '234 E PARK ST': Non-successful status code 503
Not found: 4403 15TH ST STE 204
Not found: 4403 15TH ST STE 204
Not found: 261 E COLORADO BLVD STE 21
Not found: 261 E COLORADO BLVD STE 21
Not found: 10461 MILL RUN CIR STE 810
Not found: 10461 MILL RUN CIR STE 810
Not found: 10461 MILL RUN CIR STE 810
Error geocoding '129 E 21ST ST': Non-successful status code 503
Error geocoding '129 E 21ST ST': Non-successful status code 503


In [25]:
import pandas as pd
import numpy as np

geomod_df = pd.read_csv("/kaggle/working/geomod.csv")

display(geomod_df.head(20))

df = geomod_df

import pandas as pd

# Assuming your DataFrame is called df

# Make sure NaN values are really NaN (not strings)
df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')

# Rows where both latitude and longitude are NaN
df_dropped = df[df['latitude'].isna() & df['longitude'].isna()]

# Rows where both latitude and longitude are NOT NaN
df_clean = df.dropna(subset=['latitude', 'longitude']).copy()

# Now df_clean has NO NaNs in latitude/longitude; df_dropped has ONLY NaN lat/lon
display(df_clean.head(10))

print('SHAPE:',df_clean.shape)

import folium

# Center map on Lucas County, Ohio (rough center coordinates)
m = folium.Map(location=[41.6975, -83.5439], zoom_start=11)

# Loop through df_clean and add a marker for each
for idx, row in df_clean.iterrows():
    if not pd.isna(row['latitude']) and not pd.isna(row['longitude']):
        popup_text = f"{row['full_address']}"
        folium.Marker(
            location=[row['latitude'], row['longitude']],
            popup=popup_text,
        ).add_to(m)

# Save to an HTML file to view in browser
m.save('lucas_county_map.html')

def categorize_direction(lat, lon,
                        north=41.82, south=41.48, west=-83.81, east=-83.38):
    mid_lat = (north + south) / 2
    mid_lon = (west + east) / 2
    if lat >= mid_lat:
        if lon >= mid_lon:
            return 'NE'
        else:
            return 'NW'
    else:
        if lon >= mid_lon:
            return 'SE'
        else:
            return 'SW'

df_clean['direction'] = df_clean.apply(
    lambda row: categorize_direction(row['latitude'], row['longitude']),
    axis=1
)


df_clean['direction']

df_clean

/kaggle/working/lucas_county_map.html


import folium
from folium.plugins import MarkerCluster # for clustering the markers

map = folium.Map(location=[41.6975, -83.5439], default_zoom_start=12)

import geopandas as gpd
import folium

# Option A: Download Koordinates GeoJSON layer (e.g., "lucas_county_zip_codes.geojson")
zips = gpd.read_file('lucas_county_zip_codes.geojson')

# Or Option B: Use Census TIGER/Line national ZCTA shapefile,
# then clip to Lucas County boundary.

# Sample visualization with folium
m = folium.Map(location=[41.65, -83.55], zoom_start=11)
folium.GeoJson(
    zips,
    name='ZCTAs',
    tooltip=folium.GeoJsonTooltip(fields=['ZCTA5CE10'], aliases=['ZIP Code:'])
).add_to(m)
m.save('lucas_zips_map.html')

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,parcel,owner,address,city,county,state,street_number,street_name,full_address,latitude,longitude
0,18-00065,111 SOUTH SUMMIT LLC,60 ELMORA AVE,Toledo,Lucas,Ohio,60,ELMORA AVE,"60 ELMORA AVE, Toledo, Ohio",40.665997,-74.305809
1,18-71537,111 SOUTH SUMMIT LLC,60 ELMORA AVE,Toledo,Lucas,Ohio,60,ELMORA AVE,"60 ELMORA AVE, Toledo, Ohio",40.665997,-74.305809
2,98-13717,11239 WATERSVILLE STREET L,445 CENTRAL AVE STE 304,Toledo,Lucas,Ohio,445,CENTRAL AVE STE 304,"445 CENTRAL AVE STE 304, Toledo, Ohio",NaN,NaN
3,15-00193,1301 ADAMS STREET LLC AN OCOMPANY,1301 N SUMMIT ST,Toledo,Lucas,Ohio,1301,N SUMMIT ST,"1301 N SUMMIT ST, Toledo, Ohio",41.659345,-83.521218
4,15-50334,1301 ADAMS STREET LLC AN OCOMPANY,1301 N SUMMIT ST,Toledo,Lucas,Ohio,1301,N SUMMIT ST,"1301 N SUMMIT ST, Toledo, Ohio",41.659345,-83.521218
5,15-50344,1301 ADAMS STREET LLC AN OCOMPANY,1301 N SUMMIT ST,Toledo,Lucas,Ohio,1301,N SUMMIT ST,"1301 N SUMMIT ST, Toledo, Ohio",41.659345,-83.521218
6,15-50354,1301 ADAMS STREET LLC AN OCOMPANY,1301 N SUMMIT ST,Toledo,Lucas,Ohio,1301,N SUMMIT ST,"1301 N SUMMIT ST, Toledo, Ohio",41.659345,-83.521218
7,09-06514,1322 STARR AVE LLC,2144 57TH ST,Toledo,Lucas,Ohio,2144,57TH ST,"2144 57TH ST, Toledo, Ohio",NaN,NaN
8,22-88839,1333 MATZINGER ROAD LLC,5857 FISHER RD,Toledo,Lucas,Ohio,5857,FISHER RD,"5857 FISHER RD, Toledo, Ohio",43.061328,-76.041300
9,03-15535,1421 WHITE STREET LLC,638 WILLARD ST,Toledo,Lucas,Ohio,638,WILLARD ST,"638 WILLARD ST, Toledo, Ohio",41.638791,-83.513613


,parcel,owner,address,city,county,state,street_number,street_name,full_address,latitude,longitude
0,18-00065,111 SOUTH SUMMIT LLC,60 ELMORA AVE,Toledo,Lucas,Ohio,60,ELMORA AVE,"60 ELMORA AVE, Toledo, Ohio",40.665997,-74.305809
1,18-71537,111 SOUTH SUMMIT LLC,60 ELMORA AVE,Toledo,Lucas,Ohio,60,ELMORA AVE,"60 ELMORA AVE, Toledo, Ohio",40.665997,-74.305809
3,15-00193,1301 ADAMS STREET LLC AN OCOMPANY,1301 N SUMMIT ST,Toledo,Lucas,Ohio,1301,N SUMMIT ST,"1301 N SUMMIT ST, Toledo, Ohio",41.659345,-83.521218
4,15-50334,1301 ADAMS STREET LLC AN OCOMPANY,1301 N SUMMIT ST,Toledo,Lucas,Ohio,1301,N SUMMIT ST,"1301 N SUMMIT ST, Toledo, Ohio",41.659345,-83.521218
5,15-50344,1301 ADAMS STREET LLC AN OCOMPANY,1301 N SUMMIT ST,Toledo,Lucas,Ohio,1301,N SUMMIT ST,"1301 N SUMMIT ST, Toledo, Ohio",41.659345,-83.521218
6,15-50354,1301 ADAMS STREET LLC AN OCOMPANY,1301 N SUMMIT ST,Toledo,Lucas,Ohio,1301,N SUMMIT ST,"1301 N SUMMIT ST, Toledo, Ohio",41.659345,-83.521218
8,22-88839,1333 MATZINGER ROAD LLC,5857 FISHER RD,Toledo,Lucas,Ohio,5857,FISHER RD,"5857 FISHER RD, Toledo, Ohio",43.061328,-76.041300
9,03-15535,1421 WHITE STREET LLC,638 WILLARD ST,Toledo,Lucas,Ohio,638,WILLARD ST,"638 WILLARD ST, Toledo, Ohio",41.638791,-83.513613
10,03-15537,1421 WHITE STREET LLC,638 WILLARD ST,Toledo,Lucas,Ohio,638,WILLARD ST,"638 WILLARD ST, Toledo, Ohio",41.638791,-83.513613
11,06-08311,1460 GOODALE LLC,126 WELLBROOK AVE,Toledo,Lucas,Ohio,126,WELLBROOK AVE,"126 WELLBROOK AVE, Toledo, Ohio",40.602161,-74.130619


SHAPE: (803, 11)


,parcel,owner,address,city,county,state,street_number,street_name,full_address,latitude,longitude,direction
0,18-00065,111 SOUTH SUMMIT LLC,60 ELMORA AVE,Toledo,Lucas,Ohio,60,ELMORA AVE,"60 ELMORA AVE, Toledo, Ohio",40.665997,-74.305809,SE
1,18-71537,111 SOUTH SUMMIT LLC,60 ELMORA AVE,Toledo,Lucas,Ohio,60,ELMORA AVE,"60 ELMORA AVE, Toledo, Ohio",40.665997,-74.305809,SE
3,15-00193,1301 ADAMS STREET LLC AN OCOMPANY,1301 N SUMMIT ST,Toledo,Lucas,Ohio,1301,N SUMMIT ST,"1301 N SUMMIT ST, Toledo, Ohio",41.659345,-83.521218,NE
4,15-50334,1301 ADAMS STREET LLC AN OCOMPANY,1301 N SUMMIT ST,Toledo,Lucas,Ohio,1301,N SUMMIT ST,"1301 N SUMMIT ST, Toledo, Ohio",41.659345,-83.521218,NE
5,15-50344,1301 ADAMS STREET LLC AN OCOMPANY,1301 N SUMMIT ST,Toledo,Lucas,Ohio,1301,N SUMMIT ST,"1301 N SUMMIT ST, Toledo, Ohio",41.659345,-83.521218,NE
...,...,...,...,...,...,...,...,...,...,...,...,...
945,09-67111,CAMPBELL TYRONE J & MELISS,2701 122ND ST,Toledo,Lucas,Ohio,2701,122ND ST,"2701 122ND ST, Toledo, Ohio",41.671621,-87.688016,NW
946,09-67114,CAMPBELL TYRONE J & MELISS,2701 122ND ST,Toledo,Lucas,Ohio,2701,122ND ST,"2701 122ND ST, Toledo, Ohio",41.671621,-87.688016,NW
947,83-89407,CAMPER DARYL J ETAL,3072 SHORELAND AVE,Toledo,Lucas,Ohio,3072,SHORELAND AVE,"3072 SHORELAND AVE, Toledo, Ohio",41.729988,-83.473392,NE
948,10-08511,CAN'T KEEP STILL,1318 HOAG ST,Toledo,Lucas,Ohio,1318,HOAG ST,"1318 HOAG ST, Toledo, Ohio",41.653203,-83.572158,NE


In [2]:
import geopandas as gpd
import pandas as pd
import folium

# Read GeoJSON shapefile for Lucas County zip codes
zip_gdf = gpd.read_file("lucas_county_zip_codes.geojson")

# Let's say df_clean['zip_code'] exists and has property data
zip_stats = df_clean.groupby('zip_code').size().reset_index(name='property_count')



DriverError: Failed to open dataset (flags=68): lucas_county_zip_codes.geojson